# Final results and analysis

North star: produce reproducible, language-pack-first grounded QA evidence with citation-aware outputs and transparent trust signals.

This notebook consolidates evaluation and ablation artifacts into analysis tables that can be reused for reporting.

In [8]:
from pathlib import Path

import duckdb
import pandas as pd

root = Path.cwd().resolve()
if not (root / 'artifacts').exists() and (root.parent / 'artifacts').exists():
    root = root.parent

eval_path = root / 'artifacts' / 'runs' / 'eval_results.parquet'
ablation_path = root / 'artifacts' / 'tables' / 'ablation_results.parquet'
output_dir = root / 'artifacts' / 'tables'
output_dir.mkdir(parents=True, exist_ok=True)

print('Project root:', root)
print('Eval exists:', eval_path.exists())
print('Ablation exists:', ablation_path.exists())

if not eval_path.exists() or not ablation_path.exists():
    raise FileNotFoundError(
        'Required artifacts are missing. Run 50_eval_harness.ipynb and 70_ablations.ipynb first.'
    )

con = duckdb.connect()

Project root: /Users/aaliyan/aaliyan/polyglot-grounded-qa
Eval exists: True
Ablation exists: True


## 1) Evaluation KPI summary

These metrics summarize run stability and answer behavior using the latest eval artifact.

In [2]:
eval_overview = con.execute(
    f"""
    SELECT
        run_name,
        language,
        config_hash,
        COUNT(*) AS question_count,
        SUM(CASE WHEN abstained THEN 1 ELSE 0 END) AS abstained_count,
        AVG(citation_count)::DOUBLE AS avg_citations
    FROM read_parquet('{eval_path}')
    GROUP BY run_name, language, config_hash
    ORDER BY run_name, language
    """
).df()

eval_overview['abstain_rate'] = (
    eval_overview['abstained_count'] / eval_overview['question_count']
).fillna(0.0)

eval_overview

,run_name,language,config_hash,question_count,abstained_count,avg_citations,abstain_rate
0,local-notebook-run,base,4fb4b680c55ff04f07aa6986c91ed7629909139c4b4aed...,2,0.0,1.0,0.0


## 2) Ablation comparison

Compare baseline vs no-rerank using abstention and citation behavior as trust-layer proxies.

In [3]:
ablation_summary = con.execute(
    f"""
    SELECT
        run_name,
        language,
        variant,
        COUNT(*) AS sample_count,
        SUM(CASE WHEN abstained THEN 1 ELSE 0 END) AS abstained_count,
        AVG(citation_count)::DOUBLE AS avg_citations
    FROM read_parquet('{ablation_path}')
    GROUP BY run_name, language, variant
    ORDER BY run_name, language, variant
    """
).df()

ablation_summary['abstain_rate'] = (
    ablation_summary['abstained_count'] / ablation_summary['sample_count']
).fillna(0.0)

pivot = ablation_summary.pivot(
    index=['run_name', 'language'],
    columns='variant',
    values=['abstain_rate', 'avg_citations'],
)

pivot.columns = [
    '_'.join([str(part) for part in col if str(part) != '']).strip('_')
    for col in pivot.columns.to_flat_index()
]

if 'abstain_rate_baseline' in pivot.columns and 'abstain_rate_no_rerank' in pivot.columns:
    pivot['delta_abstain_rate_no_rerank_minus_baseline'] = (
        pivot['abstain_rate_no_rerank'] - pivot['abstain_rate_baseline']
    )

if 'avg_citations_baseline' in pivot.columns and 'avg_citations_no_rerank' in pivot.columns:
    pivot['delta_avg_citations_no_rerank_minus_baseline'] = (
        pivot['avg_citations_no_rerank'] - pivot['avg_citations_baseline']
    )

ablation_summary, pivot

(             run_name language    variant  sample_count  abstained_count  \
 0  local-notebook-run     base   baseline             1              0.0   
 1  local-notebook-run     base  no_rerank             1              0.0   
 
    avg_citations  abstain_rate  
 0            1.0           0.0  
 1            1.0           0.0  ,
                              abstain_rate_baseline  abstain_rate_no_rerank  \
 run_name           language                                                  
 local-notebook-run base                        0.0                     0.0   
 
                              avg_citations_baseline  avg_citations_no_rerank  \
 run_name           language                                                    
 local-notebook-run base                         1.0                      1.0   
 
                              delta_abstain_rate_no_rerank_minus_baseline  \
 run_name           language                                                
 local-notebook-run base  

## 3) Reproducibility diagnostics and exports

Persist final tables for downstream reporting and confirm whether eval/ablation share the same config hash.

In [5]:
eval_hashes = set(
    con.execute(f"SELECT DISTINCT config_hash FROM read_parquet('{eval_path}')").fetchdf()['config_hash']
    .tolist()
)
ablation_hashes = set(
    con.execute(f"SELECT DISTINCT config_hash FROM read_parquet('{ablation_path}')").fetchdf()['config_hash']
    .tolist()
)

shared_hashes = eval_hashes.intersection(ablation_hashes)
hash_alignment = len(shared_hashes) > 0

eval_overview.to_parquet(output_dir / 'final_eval_overview.parquet', index=False)
ablation_summary.to_parquet(output_dir / 'final_ablation_summary.parquet', index=False)
pivot.to_parquet(output_dir / 'final_ablation_deltas.parquet', index=False)

diagnostics = pd.DataFrame(
    [
        {'metric': 'eval_config_hash_count', 'value': str(len(eval_hashes))},
        {'metric': 'ablation_config_hash_count', 'value': str(len(ablation_hashes))},
        {'metric': 'shared_config_hash_count', 'value': str(len(shared_hashes))},
        {'metric': 'config_hash_aligned', 'value': str(hash_alignment)},
    ]
)
diagnostics.to_parquet(output_dir / 'final_repro_diagnostics.parquet', index=False)

diagnostics

,metric,value
0,eval_config_hash_count,1
1,ablation_config_hash_count,1
2,shared_config_hash_count,1
3,config_hash_aligned,True


## 4) Finetune evaluation bridge

Load finetune-vs-baseline evaluation summaries produced by `scripts/run_finetune_eval.py`. This section is the direct comparison entrypoint for tuned adapters.

In [9]:
finetune_summary_path = root / 'artifacts' / 'tables' / 'finetune_eval_summary.parquet'
finetune_by_language_path = root / 'artifacts' / 'tables' / 'finetune_eval_by_language.parquet'

if finetune_summary_path.exists():
    finetune_summary = con.execute(
        f"SELECT * FROM read_parquet('{finetune_summary_path}') ORDER BY timestamp_utc DESC"
    ).df()
    print('Finetune summary available')
    display(finetune_summary)

    finetune_summary.to_parquet(output_dir / 'final_finetune_eval_summary.parquet', index=False)

    if finetune_by_language_path.exists():
        finetune_by_language = con.execute(
            f"SELECT * FROM read_parquet('{finetune_by_language_path}') ORDER BY language"
        ).df()
        display(finetune_by_language)
        finetune_by_language.to_parquet(
            output_dir / 'final_finetune_eval_by_language.parquet', index=False
        )
else:
    print('No finetune eval artifacts yet. Run scripts/run_finetune_eval.py first.')

Finetune summary available


,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1
0,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,29,1.000000,1.0,1.0,1.000000
1,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,29,0.793103,0.0,0.0,0.035216
2,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,29,0.793103,0.0,0.0,0.035216


,variant,timestamp_utc,language,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1
0,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,base,6,0.666667,0.0,0.0,0.041667
1,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,base,6,0.666667,0.0,0.0,0.041667
2,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,base,6,1.000000,1.0,1.0,1.000000
3,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,es,6,1.000000,0.0,0.0,0.023706
4,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,es,6,1.000000,0.0,0.0,0.023706
5,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,es,6,1.000000,1.0,1.0,1.000000
6,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,es-MX,6,0.666667,0.0,0.0,0.028913
7,baseline-pipeline,2026-03-31T18:41:49.723456+00:00,es-MX,6,0.666667,0.0,0.0,0.028913
8,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,es-MX,6,1.000000,1.0,1.0,1.000000
9,baseline-pipeline,2026-03-31T18:39:20.595319+00:00,fr,4,1.000000,0.0,0.0,0.063889


## 5) Baseline vs tuned deltas

When multiple variants exist in `finetune_eval_summary.parquet`, compute metric deltas against the baseline variant.

In [10]:
if finetune_summary_path.exists():
    history = con.execute(
        f"SELECT * FROM read_parquet('{finetune_summary_path}') ORDER BY timestamp_utc DESC"
    ).df()

    baseline_rows = history[history['variant'] == 'baseline-pipeline']
    comparison_rows = history[history['variant'] != 'baseline-pipeline']

    if not baseline_rows.empty and not comparison_rows.empty:
        baseline = baseline_rows.iloc[0]
        latest_by_variant = comparison_rows.sort_values('timestamp_utc', ascending=False).groupby('variant').head(1)

        deltas = latest_by_variant.copy()
        for metric in [
            'abstain_accuracy',
            'avg_citation_precision',
            'avg_citation_recall',
            'avg_answer_token_f1',
        ]:
            deltas[f'delta_{metric}'] = deltas[metric] - float(baseline[metric])

        cols = [
            'variant',
            'timestamp_utc',
            'rows',
            'abstain_accuracy',
            'avg_citation_precision',
            'avg_citation_recall',
            'avg_answer_token_f1',
            'delta_abstain_accuracy',
            'delta_avg_citation_precision',
            'delta_avg_citation_recall',
            'delta_avg_answer_token_f1',
        ]
        deltas = deltas[cols]
        display(deltas)
        deltas.to_parquet(output_dir / 'final_finetune_eval_deltas.parquet', index=False)
    else:
        print('Need both baseline and tuned variants to compute deltas.')
else:
    print('No finetune summary parquet found.')

,variant,timestamp_utc,rows,abstain_accuracy,avg_citation_precision,avg_citation_recall,avg_answer_token_f1,delta_abstain_accuracy,delta_avg_citation_precision,delta_avg_citation_recall,delta_avg_answer_token_f1
0,oracle-upper-bound,2026-03-31T18:41:50.298218+00:00,29,1.0,1.0,1.0,1.0,0.206897,1.0,1.0,0.964784
